In [ ]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [ ]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, coalesce, date_format
from pyspark.sql.window import Window

In [ ]:
spark = get_sparkSession("Load gold.fact_payments")

In [ ]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

In [ ]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [ ]:
_schema_name = "gold"
_table_name = "fact_payments"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_fact_payments.ipynb"
_silver_table = "silver.fact_payments"
_bad_table = f"bad_{_table_name}"

print(f"SPARK-APP : Loading started for {_fullname}")

In [ ]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [ ]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)


In [ ]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

In [ ]:
#Adding audit columns

df = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df.count()}")

# Generating surrogate key and business key

df_sk = df.withColumn("payment_sk", xxhash64(concat_ws("||", "payment_id")).cast("bigint")) \
          .withColumn("customer_bk", xxhash64(concat_ws("||", "customer_id", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)

In [ ]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_payment_date = spark.read.table("gold.dim_date").select("date_sk","date")
df_gold_customer = spark.read.table("gold.dim_customer").where("is_current = true").select("customer_sk", "customer_bk")
df_gold_currency = spark.read.table("gold.dim_currency").select("currency_sk", "currency_code")

df_invalid = df_sk.join(df_gold_customer, how = 'left_anti', on=df_sk.customer_bk==df_gold_customer.customer_bk)

print(f"SPARK-APP: Bad Record Count : {df_invalid.count()}")  

df_invalid.show(truncate = False)

if df_invalid.count() > 0:
    df_invalid.withColumn('error_description', lit("Customer not found in gold.dim_customer")) \
                       .select("payment_id", "payment_date", "order_id", "customer_id", "store", "channel", "currency", "payment_method", 
                               "payment_status", "amount", "is_refund", "refund_amount", "source_file", "created_on", "created_by", 
                               "modified_on", "modified_by", "error_description") \
                       .write \
                       .format("delta") \
                       .mode("append") \
                       .saveAsTable(f"bad.{_bad_table}")
    
df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_customer, how = "inner", on=df_sk.customer_bk==df_gold_customer.customer_bk) \
                 .join(df_gold_currency, how = "inner", on=df_sk.currency==df_gold_currency.currency_code) \
                 .join(df_payment_date, how = "left", on=df_sk.payment_date==df_payment_date.date) \
                 .select("payment_sk", "payment_id", coalesce(df_payment_date.date_sk,date_format(df_sk.payment_date, "yyyyMMdd").cast("int")).alias("payment_date"),
                         "order_id", "customer_sk", "store_sk", "channel_sk", "currency_sk", "payment_method", "payment_status", "amount", "is_refund", "refund_amount",
                         "source_file", "created_on", "created_by", "modified_on", "modified_by")

df_silver.show(truncate = False)

In [ ]:
# Truncate table for full load
dt_fact_payments = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_fact_payments.delete()
    dt_fact_payments.vacuum(0)


In [ ]:
# Merge
dt_fact_payments.alias("t").merge(
    df_silver.alias("s"),
    "t.payment_sk = s.payment_sk"
).whenMatchedUpdate(set = {   
    "payment_method":"s.payment_method",
    "payment_status":"s.payment_status",
    "amount":"s.amount",
    "is_refund":"s.is_refund",
    "refund_amount":"s.refund_amount",
    "source_file" : "s.source_file",
    "modified_on" : "s.modified_on",
    "modified_by" : "s.modified_by"
    }).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed") 

In [ ]:
hist = dt_fact_payments.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


In [ ]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

In [ ]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

In [ ]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

In [ ]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

In [ ]:
spark.stop()